# Real Dataset Preprocessing

This notebook loads the MovieLens dataset, performs k-core filtering, and prepares the matrices for our matrix completion algorithms. Since we do not have the "true" full matrix for real datasets, we must create a train/test split by taking the observed ratings and artificially hiding a portion of them (e.g., 10%) to test our predictions against.

In [ ]:
import pandas as pd
import numpy as np
import os

data_dir = '.'
ratings = pd.read_csv(os.path.join(data_dir, 'ratings.csv'))
print('Original dataset size:', ratings.shape)

In [ ]:
k = 10

# 1. Filter users with less than k ratings
user_counts = ratings['userId'].value_counts()
valid_users = user_counts[user_counts >= k].index
ratings = ratings[ratings['userId'].isin(valid_users)]

# 2. Filter movies with less than k ratings
movie_counts = ratings['movieId'].value_counts()
valid_movies = movie_counts[movie_counts >= k].index
ratings = ratings[ratings['movieId'].isin(valid_movies)]

print(f'Filtered dataset size (k={k}):', ratings.shape)

In [ ]:
# Pivot the dataframe into a 2D matrix (rows=userId, cols=movieId)
matrix_df = ratings.pivot(index='userId', columns='movieId', values='rating')

# M_full contains all the known ratings (missing values filled with 0.0)
M_full = matrix_df.fillna(0.0).values

# mask_full is True where we have a rating, False otherwise
mask_full = ~np.isnan(matrix_df.values)

print('Matrix shape:', M_full.shape)
print(f'Sparsity: {mask_full.mean():.2%} observed')

In [ ]:
# Create a train/test split for evaluation
test_ratio = 0.1

rng = np.random.default_rng(42)
train_mask = mask_full.copy()

# Get coordinates of all observed entries
observed_indices = np.argwhere(mask_full)

# Select random entries to hide for the test set
num_test = int(len(observed_indices) * test_ratio)
test_choices = rng.choice(len(observed_indices), size=num_test, replace=False)
test_indices = observed_indices[test_choices]

# Hide them in the train mask
for i, j in test_indices:
    train_mask[i, j] = False

# M_train is what the algorithm will see (test entries set to 0.0)
M_train = np.where(train_mask, M_full, 0.0)

print('Train entries:', train_mask.sum())
print('Test entries (hidden truth):', len(test_indices))

In [ ]:
out_dir = '../code/data'
os.makedirs(out_dir, exist_ok=True)

# Save these arrays to disk so they can be loaded easily into your other notebook
np.save(os.path.join(out_dir, 'M_full.npy'), M_full)          # The 'True' matrix (for evaluation only)
np.save(os.path.join(out_dir, 'mask_full.npy'), mask_full)    # Tracks all available data
np.save(os.path.join(out_dir, 'M_train.npy'), M_train)        # The 'Observed' data fed to the algorithm
np.save(os.path.join(out_dir, 'train_mask.npy'), train_mask)  # Mask showing which entries the algorithm can see

print(f'Successfully saved matrices to {out_dir}')